<a href="https://colab.research.google.com/github/lhammach/DP-representations/blob/main/dpsgd_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CKA comparison of ResNet18 on CIFAR10 - Baseline vs. Differential Privacy (DP-SGD with Opacus)

In [1]:
import torch
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

CUDA disponible : True
Nom du GPU : NVIDIA GeForce RTX 2080 Ti


## 0. Setup environment
This section installs dependencies and downloads CIFAR10 dataset.
We ensure reproducibility across baseline and DP training.

## 0.1 Google Colab setup
We download CIFAR10 dataset from FastAI repository and extract it.
This dataset is used for both baseline and DP experiments.

In [2]:
import os
import tarfile
import urllib.request

data_dir = "cifar10"
archive_name = "cifar10.tgz"
url = "https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz"

if not os.path.exists(data_dir):
    print("Téléchargement de CIFAR-10 en cours (160 Mo)...")
    urllib.request.urlretrieve(url, archive_name)
    
    print("Extraction des images en cours...")
    with tarfile.open(archive_name, "r:gz") as tar:
        tar.extractall()
        
    os.remove(archive_name)  # Supprime immédiatement l'archive pour libérer de l'espace
    print("Terminé ! Le dataset est prêt et le dossier est propre.")
else:
    print("Le dossier 'cifar10' existe déjà. Étape ignorée.")

Le dossier 'cifar10' existe déjà. Étape ignorée.


## 0.2 Imports
We import torchvision and Opacus.
Opacus is used only for the differential privacy training pipeline (DP-SGD).

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms

from torchvision.datasets import CIFAR10
from torchvision.datasets import ImageFolder
from torchvision import models
import torch.nn as nn
import torch.optim as optim

import opacus
from opacus import PrivacyEngine

import numpy as np
from opacus.utils.batch_memory_manager import BatchMemoryManager

from torchvision.models.resnet import BasicBlock

from tqdm import tqdm 

from pathlib import Path

# Création du répertoire pour sauvegarder les réseaux entraînés
NETWORKS_DIR = Path("./networks")
NETWORKS_DIR.mkdir(parents=True, exist_ok=True)

def get_checkpoint_path(prefix, epsilon, seed, save_dir=NETWORKS_DIR, ext="pth"):
    i = 0
    while True:
        path = save_dir / f"{prefix}_eps{int(epsilon)}_seed{seed}_i{i}.{ext}"
        if not path.exists():
            return path
        i += 1

In [4]:
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("opacus:", opacus.__version__)

torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
opacus: 1.6.0


## 1. Hyperparameters
We define training and DP-related hyperparameters:

- EPOCHS: number of training epochs (shared between baseline and DP)
- LR: learning rate
- EPSILON, DELTA: privacy budget parameters for DP-SGD
- MAX_GRAD_NORM: gradient clipping bound
- BATCH_SIZE: logical batch size
- MAX_PHYSICAL_BATCH_SIZE: memory constraint for DP training

For reproducibility, we also fix a global random seed.

> **Note on `DELTA`.**  
> We set  
> $$
> \delta << \frac{1}{N}
> $$
> where $N$ is the number of training examples.  

In [5]:
# hyperparamètres
MAX_GRAD_NORM = 1.2
EPSILON = 10.0
DELTA = 1e-8 # << 1 / len(train_dataset)
EPOCHS = 5

LR = 1e-3

# fixer la seed GLOBALEMENT pour la reproductibilité
SEED = 42

import random
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

With DP-SGD, the training requires more memory than with a classical training because we need to store each gradient of each sample (per-sample gradient) before we clip it. We thus use `BatchMemoryManager` which enables to process (MAX_PHYSICAL_BATCH_SIZE =128) examples at a time instead of (BATCH_SIZE = 512) at once.

In [6]:
BATCH_SIZE = 128 # 512
MAX_PHYSICAL_BATCH_SIZE = 32 # 128

# 2. Dataset : CIFAR10

### Data processing
The CIFAR10 images are initially PIL images. We process the into centered and normalized tensors (3 × 32 × 32).
We use CIFAR10 dataset with standard normalization.
No data augmentation is applied to avoid utility degradation in DP setting.

The same CIFAR10 train/test split is used for:

- baseline training
- DP-SGD training
- CKA comparison

To make the comparison fair, both the baseline and the DP model will use the **same architecture** and the **same preprocessing pipeline**.  
The only difference between the two training procedures is the use (or not) of differential privacy during optimization.

We load CIFAR10 using `ImageFolder` and create two dataloaders:

- a **training dataloader** used for both baseline and DP training
- a **test dataloader** used for evaluation

In [ ]:
# These values, specific to the CIFAR10 dataset, are assumed to be known.
# If necessary, they can be computed with modest privacy budgets.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD_DEV = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD_DEV),
])

# Loading CIFAR10 (avec ImageFolder)
DATA_ROOT = './cifar10'
train_dataset = ImageFolder(root='./cifar10/train', transform=transform)
test_dataset = ImageFolder(root='./cifar10/test', transform=transform)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    drop_last=True, # Added to ensure consistent batch sizes
    # shuffle=True,
    # num_workers=4,
    # pin_memory=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False, #on met false pour utiliser une seule classe de test
    # num_workers=4,
    # pin_memory=True
)

print(f"Dataset prêt !")
print(f"Nombre d'images d'entraînement : {len(train_dataset)}")
print(f"Nombre d'images de test : {len(test_dataset)}")

image, label = train_dataset[0]
print(image.shape)
print(label)

Dataset prêt !
Nombre d'images d'entraînement : 50000
Nombre d'images de test : 10000
torch.Size([3, 32, 32])
0


## 3. Models

We use the same `ResNet18` architecture for both experiments:

- a **baseline model** trained with standard optimization
- a **DP model** trained with DP-SGD via Opacus

To ensure a fair comparison for the CKA analysis, **both models use the same DP-compatible architecture**.

> Why modify the original ResNet18?
>
> The default torchvision `ResNet18` contains components that are not ideal for Opacus:
> - **BatchNorm** layers are replaced with **GroupNorm**
> - **inplace ReLU** operations are disabled
> - the residual block forward pass is patched to avoid inplace residual additions

These modifications are applied **to both the baseline and the DP model**, so that the only difference between the two experiments is the training procedure itself. We use ResNet18 architecture for both:
- Baseline model (standard SGD)
- DP model (DP-SGD via Opacus)

To ensure compatibility with DP-SGD:
- BatchNorm layers are replaced by GroupNorm
- ReLU inplace operations are disabled
- Residual connections are made non-inplace-safe

In [8]:
# savoir si gpu disponible
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

# utilise le GPU si disponible, sinon le CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA disponible : True
Nom du GPU : NVIDIA GeForce RTX 2080 Ti


### 3.1 Building a DP-compatible ResNet18

We define a helper function that creates a ResNet18 compatible with Opacus.  
This function will be used for **both** the baseline and the DP model.

In [9]:
from torchvision.models.resnet import BasicBlock
from opacus.validators import ModuleValidator


def safe_basicblock_forward(self, x):
    identity = x

    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    if self.downsample is not None:
        identity = self.downsample(x)

    # no inplace residual addition
    out = out + identity
    out = self.relu(out)

    return out


def make_resnet18_opacus_compatible(num_classes=10):
    """
    Build a ResNet18 compatible with Opacus:
    - residual addition made non-inplace-safe
    - BatchNorm replaced with GroupNorm
    - all ReLU set to inplace=False
    """
    # Patch BasicBlock forward globally
    BasicBlock.forward = safe_basicblock_forward

    model = models.resnet18(num_classes=num_classes)

    # Replace BatchNorm layers with GroupNorm layers
    model = ModuleValidator.fix(model)

    # Disable inplace ReLU everywhere
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False

    # Validate compatibility
    errors = ModuleValidator.validate(model, strict=False)
    if len(errors) > 0:
        print("Opacus compatibility warnings:")
        for err in errors:
            print(err)

    return model

### 3.2 Instantiating the baseline and DP models

We now create two separate models from the same architecture:

- `baseline_model`: trained without privacy
- `dp_model`: trained with DP-SGD

Both start from the same ResNet18 design, which makes the later CKA comparison more meaningful.

In [10]:
baseline_model = make_resnet18_opacus_compatible(num_classes=10).to(device)
dp_model = make_resnet18_opacus_compatible(num_classes=10).to(device)

baseline_optimizer = optim.RMSprop(baseline_model.parameters(), lr=LR)
dp_optimizer = optim.RMSprop(dp_model.parameters(), lr=LR)

print("Baseline model:")
print(baseline_model)

print("\nDP model (before Opacus wrapping):")
print(dp_model)

print("Validation baseline:", ModuleValidator.validate(baseline_model, strict=False))
print("Validation DP:", ModuleValidator.validate(dp_model, strict=False))

Baseline model:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): GroupNorm(32, 64, eps=1e-05, affine=True)
  (relu): ReLU()
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(32, 64, eps=1e-05, affine=True)
      (relu): ReLU()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(32, 64, eps=1e-05, affine=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(32, 64, eps=1e-05, affine=True)
      (relu): ReLU()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(32, 64, eps=1e-05, affine=True)
    )
  )
  (layer2): Sequ

## 4. Training

We train the two models under two different regimes:

- **Baseline training**: standard optimization without privacy
- **DP training**: differentially private SGD using Opacus

Both models use:

- the **same architecture**
- the **same train/test split**
- the **same preprocessing**

The only difference is whether privacy mechanisms (gradient clipping + noise addition) are applied during training.

In [11]:
# def accuracy(preds, labels):
#     return (preds == labels).mean()

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.RMSprop(model.parameters(), lr=LR) # optimizer for GD

def accuracy_from_logits(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

### 4.1 Training the Baseline Network

In [12]:
def train_baseline(model, train_loader, optimizer, epoch, device):
    model.train()
    criterion = nn.CrossEntropyLoss()

    losses = []
    accs = []

    for images, target in train_loader:
        images = images.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, target)
        acc = accuracy_from_logits(output, target)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        accs.append(acc)

    print(
        f"[Baseline] Epoch {epoch} | "
        f"Loss: {np.mean(losses):.4f} | "
        f"Acc: {np.mean(accs) * 100:.2f}%"
    )

In [13]:
def test(model, test_loader, device, prefix="Test"):
    model.eval()
    criterion = nn.CrossEntropyLoss()

    losses = []
    accs = []

    with torch.no_grad():
        for images, target in test_loader:
            images = images.to(device)
            target = target.to(device)

            output = model(images)
            loss = criterion(output, target)
            acc = accuracy_from_logits(output, target)

            losses.append(loss.item())
            accs.append(acc)

    print(
        f"[{prefix}] "
        f"Loss: {np.mean(losses):.4f} | "
        f"Acc: {np.mean(accs) * 100:.2f}%"
    )

    return np.mean(accs)

In [14]:
for epoch in tqdm(range(EPOCHS), desc="Baseline"):
    train_baseline(baseline_model, train_loader, baseline_optimizer, epoch + 1, device)
    test(baseline_model, test_loader, device, prefix="Baseline test")

Baseline:   0%|          | 0/5 [00:00<?, ?it/s]

[Baseline] Epoch 1 | Loss: 0.3000 | Acc: 96.88%


Baseline:  20%|██        | 1/5 [01:41<06:44, 101.19s/it]

[Baseline test] Loss: 8.4525 | Acc: 11.00%
[Baseline] Epoch 2 | Loss: 0.1984 | Acc: 96.62%


Baseline:  40%|████      | 2/5 [03:19<04:58, 99.45s/it] 

...

[Baseline test] Loss: 9.4858 | Acc: 11.00%
[Baseline] Epoch 4 | Loss: 0.2191 | Acc: 95.59%


Baseline:  80%|████████  | 4/5 [06:34<01:38, 98.67s/it]

[Baseline test] Loss: 9.4233 | Acc: 11.00%
[Baseline] Epoch 5 | Loss: 0.2163 | Acc: 95.59%


Baseline: 100%|██████████| 5/5 [08:15<00:00, 99.02s/it]

[Baseline test] Loss: 9.2296 | Acc: 11.00%


In [15]:
# # save for cka_analysis.ipynb

baseline_save_path = get_checkpoint_path(
    prefix="baseline_resnet18",
    epsilon=f"{int(EPSILON)}",
    seed=f"{SEED}"
)

torch.save(
    baseline_model.state_dict(),
    baseline_save_path
)

print(f"Baseline sauvegardée dans : {baseline_save_path}")

Baseline sauvegardée dans : networks/baseline_resnet18_eps10_seed42_i0.pth


### 4.2 Training the DP model with Opacus

We now train the second model using DP-SGD through Opacus.

The DP pipeline consists of two steps:

1. **privatizing** the model, optimizer and dataloader with `PrivacyEngine`
2. training with `BatchMemoryManager` to split large logical batches into smaller physical batches that fit in memory

At the end of each epoch, we also report the privacy budget $\varepsilon$.

#### 4.2.a Preparing the model for private training

`PrivacyEngine.make_private_with_epsilon(...)` wraps:

- the model
- the optimizer
- the training dataloader

so that training becomes differentially private.

Given the target values $(\varepsilon, \delta)$, the accountant determines the noise multiplier needed to reach the requested privacy budget after the specified number of epochs.

In [16]:
privacy_engine = PrivacyEngine(accountant="rdp")

print("------ Before make_private_with_epsilon ------")
print(type(dp_model))
print(type(dp_optimizer))
print(type(train_loader))

dp_model, dp_optimizer, dp_train_loader = privacy_engine.make_private_with_epsilon(
    module=dp_model,
    optimizer=dp_optimizer,
    data_loader=train_loader,
    epochs=EPOCHS,
    target_epsilon=EPSILON,
    target_delta=DELTA,
    max_grad_norm=MAX_GRAD_NORM,
)

print(f"\nUsing sigma={dp_optimizer.noise_multiplier} and C={MAX_GRAD_NORM}")

print("\n------ After make_private_with_epsilon ------")
print(type(dp_model))
print(type(dp_optimizer))
print(type(dp_train_loader))

------ Before make_private_with_epsilon ------
<class 'torchvision.models.resnet.ResNet'>
<class 'torch.optim.rmsprop.RMSprop'>
<class 'torch.utils.data.dataloader.DataLoader'>


/home/lamsade/lhammache/DP-representations/.venv/lib/python3.10/site-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/home/lamsade/lhammache/DP-representations/.venv/lib/python3.10/site-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(
06/22/2026 23:43:57:WARNING:Ignoring drop_last as it is not compatible with DPDataLoader.



Using sigma=0.521697998046875 and C=1.2

------ After make_private_with_epsilon ------
<class 'opacus.grad_sample.grad_sample_module.GradSampleModule'>
<class 'opacus.optimizers.optimizer.DPOptimizer'>
<class 'opacus.data_loader.DPDataLoader'>


#### 4.2.b Private training loop

During DP training, Opacus computes **per-sample gradients** in order to clip them and add noise.  
This is more memory-intensive than standard training.

To reduce memory usage, we use `BatchMemoryManager`, which splits a large logical batch into smaller physical batches during training.

In [17]:
def train_dp(model, train_loader, optimizer, epoch, device, privacy_engine):
    model.train()
    criterion = nn.CrossEntropyLoss()

    losses = []
    accs = []

    with BatchMemoryManager(
        data_loader=train_loader,
        max_physical_batch_size=MAX_PHYSICAL_BATCH_SIZE,
        optimizer=optimizer
    ) as memory_safe_data_loader:

        for images, target in memory_safe_data_loader:
            images = images.to(device)
            target = target.to(device)

            optimizer.zero_grad()

            output = model(images)
            loss = criterion(output, target)
            acc = accuracy_from_logits(output, target)

            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            accs.append(acc)

    epsilon = privacy_engine.get_epsilon(DELTA)

    print(
        f"[DP] Epoch {epoch} | "
        f"Loss: {np.mean(losses):.4f} | "
        f"Acc: {np.mean(accs) * 100:.2f}% | "
        f"(ε = {epsilon:.2f}, δ = {DELTA:.2e})"
    )

In [18]:
import torch
import opacus

print("=== Versions ===")
print("torch        :", torch.__version__)
print("torchvision  :", torchvision.__version__)
print("opacus       :", opacus.__version__)

print("\n=== FSDP checks ===")
print("has torch.distributed ?", hasattr(torch, "distributed"))
print("has torch.distributed.fsdp ?", hasattr(torch.distributed, "fsdp") if hasattr(torch, "distributed") else False)

if hasattr(torch, "distributed") and hasattr(torch.distributed, "fsdp"):
    print("has FSDPModule ?", hasattr(torch.distributed.fsdp, "FSDPModule"))
    print("fsdp module =", torch.distributed.fsdp)
else:
    print("FSDP namespace not available")

=== Versions ===
torch        : 2.5.1+cu121
torchvision  : 0.20.1+cu121
opacus       : 1.6.0

=== FSDP checks ===
has torch.distributed ? True
has torch.distributed.fsdp ? True
has FSDPModule ? False
fsdp module = <module 'torch.distributed.fsdp' from '/home/lamsade/lhammache/DP-representations/.venv/lib/python3.10/site-packages/torch/distributed/fsdp/__init__.py'>


In [19]:
import types
import torch

# Temporary compatibility patch for some torch/opacus version mismatches
if hasattr(torch, "distributed") and hasattr(torch.distributed, "fsdp"):
    if not hasattr(torch.distributed.fsdp, "FSDPModule"):
        class _DummyFSDPModule:
            pass

        torch.distributed.fsdp.FSDPModule = _DummyFSDPModule
        print("Temporary patch applied: torch.distributed.fsdp.FSDPModule added.")
    else:
        print("No patch needed: FSDPModule already exists.")
else:
    print("No FSDP namespace available in torch.distributed.")

Temporary patch applied: torch.distributed.fsdp.FSDPModule added.


In [20]:
for epoch in tqdm(range(EPOCHS), desc="DP", unit="epoch"):
    train_dp(dp_model, dp_train_loader, dp_optimizer, epoch + 1, device, privacy_engine)
    test(dp_model, test_loader, device, prefix="DP test")

DP:   0%|          | 0/5 [00:00<?, ?epoch/s]

[DP] Epoch 1 | Loss: 1.9526 | Acc: 29.94% | (ε = 7.87, δ = 1.00e-08)


DP:  20%|██        | 1/5 [03:12<12:48, 192.14s/epoch]

[DP test] Loss: 1.8103 | Acc: 35.65%
[DP] Epoch 2 | Loss: 1.7757 | Acc: 38.15% | (ε = 8.59, δ = 1.00e-08)


DP:  40%|████      | 2/5 [06:23<09:35, 191.75s/epoch]

...

[DP test] Loss: 1.7813 | Acc: 41.62%
[DP] Epoch 4 | Loss: 1.7509 | Acc: 43.31% | (ε = 9.58, δ = 1.00e-08)


DP:  80%|████████  | 4/5 [12:32<03:06, 186.70s/epoch]

[DP test] Loss: 1.7885 | Acc: 42.99%
[DP] Epoch 5 | Loss: 1.7463 | Acc: 44.65% | (ε = 10.00, δ = 1.00e-08)


DP: 100%|██████████| 5/5 [15:38<00:00, 187.66s/epoch]

[DP test] Loss: 1.7896 | Acc: 43.30%


In [21]:
# # save for cka_analysis.ipynb

dp_save_path = get_checkpoint_path(
    prefix="dp_resnet18",
    epsilon=f"{int(EPSILON)}",
    seed=f"{SEED}"
)

dp_state_dict = dp_model._module.state_dict() if hasattr(dp_model, "_module") else dp_model.state_dict()

torch.save(
    dp_state_dict,
    dp_save_path
)

print(f"Modèle DP sauvegardé dans : {dp_save_path}")

Modèle DP sauvegardé dans : networks/dp_resnet18_eps10_seed42_i0.pth
